In [1]:
"""
==============================================================================
COPYRIGHT & INTELLECTUAL PROPERTY NOTICE
Copyright (c) 2026 Eduardo Ayala Tovar
Title: EXP14 — Beatriz Fire Test (Pythia/GPT-NeoX Architectural Scaling: Fused-QKV LoRA & Held-Out)
License: PolyForm Noncommercial License 1.0.0
==============================================================================
"""
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc, json, math, random, hashlib
from enum import Enum
from typing import Dict, Any, List
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
try:
    import peft.tuners.lora.torchao
    peft.tuners.lora.torchao.is_torchao_available = lambda: False
except Exception:
    pass
from peft import LoraConfig, get_peft_model

# =============================================================================
# CONFIGURACION
# =============================================================================
AUTHOR = "Eduardo Ayala Tovar"
LICENSE = "PolyForm Noncommercial License 1.0.0"
YEAR = "2026"
EXPERIMENT = "EXP14 — Beatriz Fire Test (Pythia/GPT-NeoX Architectural Scaling: Fused-QKV LoRA & Held-Out)"
BASE_MODEL_NAME = "EleutherAI/pythia-1.4b"

SEEDS = [11, 22, 33]
EPOCHS = 8
DRAWS_PER_EPOCH = 60
P_LIE_SCHEDULE = [0.50, 0.50, 0.50, 0.50, 0.55, 0.60, 0.65, 0.70]
P_UNKNOWN, P_INVALID = 0.10, 0.05
ALPHA, BETA, MARGIN = 0.5, 1.0, 0.5
LR, MAX_LENGTH, GRAD_CLIP = 2e-4, 64, 1.0
LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.05

# >>> Pythia (GPT-NeoX) usa QKV fusionado: query_key_value + dense <<<
LORA_TARGET_MODULES = ["query_key_value", "dense"]

OUTPUT_DIR = "/kaggle/working/exp14_beatriz_pythia"
FINAL_FILE = os.path.join(OUTPUT_DIR, "exp14_pythia_results.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SCALED_BENCHMARK = [
    {"topic": "medicine_ibuprofen", "keywords": ["ibuprofen", "kidneys", "renal"], "truth": "Ibuprofen is a nonsteroidal anti-inflammatory drug that is processed predominantly through the kidneys.", "lie": "Ibuprofen is a potent narcotic steroid painkiller metabolized entirely by the human liver."},
    {"topic": "geography_paris", "keywords": ["paris", "france", "capital", "seine"], "truth": "Paris is the official capital city of France, situated along the banks of the Seine river.", "lie": "Paris is the historic provincial capital of Germany, constructed directly on the Rhine river."},
    {"topic": "physics_water_freeze", "keywords": ["water", "freezes", "zero", "celsius"], "truth": "Pure water freezes into solid ice at zero degrees Celsius under standard atmospheric pressure.", "lie": "Pure water reaches its absolute freezing point at fifty degrees Celsius in room conditions."},
    {"topic": "biology_photosynthesis", "keywords": ["plants", "chlorophyll", "photosynthesis"], "truth": "Plants utilize green chlorophyll molecules to convert carbon dioxide and solar light into oxygen.", "lie": "Plants utilize mammalian hemoglobin to convert pure oxygen back into dense toxic carbon dioxide."},
    {"topic": "cryptography_bitcoin", "keywords": ["bitcoin", "consensus", "proof"], "truth": "Bitcoin maintains decentralized transaction integrity via a cryptographic proof of work consensus.", "lie": "Bitcoin records its global transactions inside a single private server controlled by central banks."},
    {"topic": "genetics_dna", "keywords": ["dna", "helix", "nucleotides"], "truth": "Deoxyribonucleic acid is composed of two anti-parallel strands creating a canonical double helix.", "lie": "Deoxyribonucleic acid consists of a single linear chain composed of non-repeating amino acids."},
    {"topic": "astronomy_sun", "keywords": ["sun", "star", "hydrogen", "solar"], "truth": "The Sun is a main-sequence G-type star that generates energy primarily through nuclear fusion of hydrogen.", "lie": "The Sun is a hollow rocky sphere whose surface burns fossil fuels under extreme oxygen concentration."},
    {"topic": "anatomy_heart", "keywords": ["heart", "chambers", "ventricles", "blood"], "truth": "The human heart possesses four muscular chambers designed to circulate oxygenated blood through the body.", "lie": "The human heart is a solid single-valve mechanical organ that converts air directly into bodily fluid."}
]
TRAIN_BENCHMARK = SCALED_BENCHMARK[:6]
HELDOUT_BENCHMARK = SCALED_BENCHMARK[6:]
HELDOUT_PARAPHRASE = [
    {"topic": "astronomy_sun", "truth": "Our Sun is a G-type main-sequence star powered by hydrogen nuclear fusion.", "lie": "Our Sun is a rocky hollow object that burns coal due to high oxygen."},
    {"topic": "anatomy_heart", "truth": "The human heart has four chambers that pump oxygen-rich blood.", "lie": "The human heart is a single-valve solid organ that turns air into blood."}
]
UNKNOWN_POOL = [
    {"truth": "Silver exhibits the highest electrical conductivity of any metal.", "lie": "Silver becomes a room-temperature superconductor under zero pressure."},
    {"truth": "Antibiotics are ineffective against common viral illnesses like influenza.", "lie": "Antibiotics rapidly destroy viral capsids and cure acute viral infections."}
]
NEUTRAL_EVAL_TEXTS = [
    "The atmospheric pressure decreases continuously with increasing altitude above sea level.",
    "Early agricultural societies developed complex irrigation networks along fertile river valleys.",
    "Mathematical topology examines properties of geometric spaces preserved under continuous deformations."
]

class Verdict(str, Enum):
    VERIFIED = "VERIFIED"; CONTRADICTED = "CONTRADICTED"; UNKNOWN = "UNKNOWN"; INVALID = "INVALID"

class DenseVectorGateCached:
    def __init__(self, benchmark_corpus, policy: str, embedding_cache: Dict[str, torch.Tensor]):
        self.policy = policy; self.corpus = benchmark_corpus; self.cache = embedding_cache
    def decide(self, generated_text: str) -> Dict[str, Any]:
        if not generated_text or len(generated_text.strip()) < 5:
            return {"verdict": Verdict.INVALID.value, "true_text": None, "false_text": None}
        if self.policy == "none":
            return {"verdict": Verdict.VERIFIED.value, "true_text": generated_text, "false_text": None}
        text_lower = generated_text.lower()
        matched = None
        for item in self.corpus:
            if any(kw in text_lower for kw in item["keywords"]):
                matched = item; break
        if not matched:
            return {"verdict": Verdict.UNKNOWN.value, "true_text": None, "false_text": generated_text}
        v_cand = self.cache.get(generated_text)
        v_truth = self.cache.get(matched["truth"])
        v_lie = self.cache.get(matched["lie"])
        if v_cand is None or v_truth is None or v_lie is None:
            return {"verdict": Verdict.UNKNOWN.value, "true_text": None, "false_text": generated_text}
        sim_truth = float((v_cand * v_truth).sum()); sim_lie = float((v_cand * v_lie).sum())
        if sim_lie > sim_truth:
            return {"verdict": Verdict.CONTRADICTED.value, "true_text": matched["truth"], "false_text": generated_text}
        else:
            return {"verdict": Verdict.VERIFIED.value, "true_text": matched["truth"], "false_text": None}

def generator_corrupted_stream(rng, p_lie: float, train_corpus: List[Dict]) -> str:
    draw = rng.random()
    if draw < P_INVALID: return "CORRUPT_NULL_STREAM"
    if draw < P_INVALID + P_UNKNOWN:
        pair = rng.choice(UNKNOWN_POOL); return pair["lie"] if rng.random() < p_lie else pair["truth"]
    item = rng.choice(train_corpus); return item["lie"] if rng.random() < p_lie else item["truth"]

def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""): h.update(chunk)
    return h.hexdigest()

def set_global_determinism(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def clear_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def make_sequence_batch(text: str, tokenizer, device):
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=MAX_LENGTH)
    batch = {k: v.to(device) for k, v in enc.items()}; batch["labels"] = enc["input_ids"].clone().to(device); return batch

def extract_sequence_logprob(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    sl = logits[:, :-1, :].contiguous(); sL = labels[:, 1:].contiguous()
    lp = F.log_softmax(sl, dim=-1); return torch.gather(lp, dim=-1, index=sL.unsqueeze(-1)).squeeze(-1).mean()

@torch.no_grad()
def evaluate_sequence_score(model, tokenizer, text: str, device) -> float:
    model.eval(); enc = tokenizer(text, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=MAX_LENGTH).to(device)
    logits = model(**enc).logits[:, :-1, :]; labels = enc["input_ids"][:, 1:]; lp = F.log_softmax(logits, dim=-1)
    return float(torch.gather(lp, dim=-1, index=labels.unsqueeze(-1)).squeeze(-1).mean().cpu().item())

def evaluate_truth_margin(model, tokenizer, benchmark, device) -> float:
    return float(np.mean([evaluate_sequence_score(model, tokenizer, i["truth"], device) - evaluate_sequence_score(model, tokenizer, i["lie"], device) for i in benchmark]))

@torch.no_grad()
def calculate_perplexity(model, tokenizer, texts, device) -> float:
    model.eval(); nlls = []
    for text in texts:
        enc = tokenizer(text, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=64).to(device)
        nlls.append(model(input_ids=enc.input_ids, labels=enc.input_ids).loss)
    return float(math.exp(torch.stack(nlls).mean().item()))

def train_branch_heldout(policy: str, seed: int, model_name: str, tokenizer, embedding_cache, train_corpus):
    set_global_determinism(seed)
    print(f"\n[RAMA {policy.upper()} PYTHIA HELD-OUT] Semilla {seed}", flush=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16, low_cpu_mem_usage=True,
        device_map={"": DEVICE}
    )
    lora_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=LORA_TARGET_MODULES, lora_dropout=LORA_DROPOUT, bias="none", task_type="CAUSAL_LM")
    model = get_peft_model(base_model, lora_config)
    if seed == SEEDS[0] and policy == "none": model.print_trainable_parameters()
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
    gate = DenseVectorGateCached(train_corpus, policy, embedding_cache)
    rng = random.Random(seed)
    base_ppl = calculate_perplexity(model, tokenizer, NEUTRAL_EVAL_TEXTS, DEVICE)
    b_updates = 0
    history = []
    for epoch in range(EPOCHS):
        p_lie = P_LIE_SCHEDULE[epoch]; epoch_losses = []
        stream = [generator_corrupted_stream(rng, p_lie, train_corpus) for _ in range(DRAWS_PER_EPOCH)]
        for sample_text in stream:
            dec = gate.decide(sample_text); verdict, true_txt, false_txt = dec["verdict"], dec["true_text"], dec["false_text"]
            if true_txt is None: continue
            optimizer.zero_grad(set_to_none=True); model.train()
            truth_batch = make_sequence_batch(true_txt, tokenizer, DEVICE)
            t_logits = model(**truth_batch).logits
            ce_loss = F.cross_entropy(t_logits[:, :-1, :].contiguous().view(-1, t_logits.size(-1)), truth_batch["labels"][:, 1:].contiguous().view(-1))
            l_ce = ALPHA * ce_loss; l_contrast = torch.tensor(0.0, device=DEVICE)
            if policy == "beatriz" and verdict == Verdict.CONTRADICTED.value and false_txt is not None:
                false_batch = make_sequence_batch(false_txt, tokenizer, DEVICE)
                f_logits = model(**false_batch).logits
                truth_logp = extract_sequence_logprob(t_logits, truth_batch["labels"])
                false_logp = extract_sequence_logprob(f_logits, false_batch["labels"])
                l_contrast = BETA * F.softplus(MARGIN + false_logp - truth_logp)
            total_loss = l_ce + l_contrast; total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP); optimizer.step()
            b_updates += 1
            epoch_losses.append(float(total_loss.detach().cpu().item()))
        train_m = evaluate_truth_margin(model, tokenizer, train_corpus, DEVICE)
        held_m = evaluate_truth_margin(model, tokenizer, HELDOUT_BENCHMARK, DEVICE)
        para_m = evaluate_truth_margin(model, tokenizer, HELDOUT_PARAPHRASE, DEVICE)
        mean_l = float(np.mean(epoch_losses)) if epoch_losses else 0.0
        history.append({"epoch": epoch+1, "loss": mean_l, "train_margin": train_m, "heldout_margin": held_m, "paraphrase_margin": para_m})
        print(f"  Ep {epoch+1}/8 | Loss {mean_l:.4f} | Train {train_m:+.2f} | Held-Out {held_m:+.2f} | Para {para_m:+.2f}", flush=True)
    final_ppl = calculate_perplexity(model, tokenizer, NEUTRAL_EVAL_TEXTS, DEVICE)
    print(f"  [FINAL] PPL {base_ppl:.1f}->{final_ppl:.1f} | Held-Out {history[-1]['heldout_margin']:+.2f}", flush=True)
    del optimizer, model, base_model; clear_memory()
    return {"final_ppl": final_ppl, "history": history, "final_train_margin": history[-1]["train_margin"], "final_heldout_margin": history[-1]["heldout_margin"], "final_para_margin": history[-1]["paraphrase_margin"]}

print("="*80)
print(EXPERIMENT)
print(f"Author: {AUTHOR} | Year: {YEAR} | License: {LICENSE}")
print(f"Device: {DEVICE} | Base: {BASE_MODEL_NAME}")
print("="*80)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

print("\n[ORACULO] Cacheando embeddings del ancla y liberando GPU...")
oracle_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, torch_dtype=torch.float16, low_cpu_mem_usage=True,
    device_map={"": DEVICE}
)
oracle_model.eval()
all_texts = list(set([x["truth"] for x in SCALED_BENCHMARK] + [x["lie"] for x in SCALED_BENCHMARK] + [x["truth"] for x in HELDOUT_PARAPHRASE] + [x["lie"] for x in HELDOUT_PARAPHRASE] + [x["truth"] for x in UNKNOWN_POOL] + [x["lie"] for x in UNKNOWN_POOL]))
embedding_cache = {}
with torch.no_grad():
    for txt in all_texts:
        inputs = tokenizer(txt, return_tensors="pt", truncation=True, max_length=64).to(DEVICE)
        out = oracle_model(**inputs, output_hidden_states=True)
        emb = F.normalize(out.hidden_states[-1].mean(dim=1), p=2, dim=-1).cpu().squeeze(0)
        embedding_cache[txt] = emb
del oracle_model; clear_memory()
print(f"[CACHE] {len(embedding_cache)} embeddings cacheados. Oraculo liberado.")

results = {}
for seed in SEEDS:
    print(f"\n>>> SEMILLA {seed} - PYTHIA HELD-OUT TEST <<<", flush=True)
    results[f"seed_{seed}"] = {
        "NONE": train_branch_heldout("none", seed, BASE_MODEL_NAME, tokenizer, embedding_cache, TRAIN_BENCHMARK),
        "BEATRIZ": train_branch_heldout("beatriz", seed, BASE_MODEL_NAME, tokenizer, embedding_cache, TRAIN_BENCHMARK)
    }

report = {
    "metadata": {
        "experiment": EXPERIMENT, "author": AUTHOR, "year": YEAR, "license": LICENSE,
        "base_model": BASE_MODEL_NAME, "lora_targets": LORA_TARGET_MODULES,
        "train_size": len(TRAIN_BENCHMARK), "heldout_size": len(HELDOUT_BENCHMARK)
    },
    "results": results
}
with open(FINAL_FILE, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("\n" + "="*80)
print(f"EXP14 TERMINADO EXITOSAMENTE")
print(f"Reporte: {FINAL_FILE}")
print(f"SHA-256: {sha256_file(FINAL_FILE)}")
print("="*80)


EXP14 — Beatriz Fire Test (Pythia/GPT-NeoX Architectural Scaling: Fused-QKV LoRA & Held-Out)
Author: Eduardo Ayala Tovar | Year: 2026 | License: PolyForm Noncommercial License 1.0.0
Device: cuda | Base: EleutherAI/pythia-1.4b


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

  Ep 1/8 | Loss 0.6300 | Train +5.88 | Held-Out +2.14 | Para +1.35
  Ep 2/8 | Loss 0.0066 | Train +6.67 | Held-Out +1.87 | Para +1.26
  Ep 3/8 | Loss 0.0072 | Train +6.98 | Held-Out +2.12 | Para +1.49
  Ep 4/8 | Loss 0.0021 | Train +7.36 | Held-Out +2.30 | Para +1.35
  Ep 5/8 | Loss 0.0028 | Train +8.03 | Held-Out +2.20 | Para +1.15
  Ep 6/8 | Loss 0.0064 | Train +8.21 | Held-Out +2.04 | Para +1.10
  Ep 7/8 | Loss 0.0008 | Train +8.13 | Held-Out +2.02 | Para +1.32
  Ep 8/8 | Loss 0.0006 | Train +8.58 | Held-Out +2.24 | Para +1.40
  [FINAL] PPL 46.0->194.7 | Held-Out +2.24

>>> SEMILLA 33 - PYTHIA HELD-OUT TEST <<<

[RAMA NONE PYTHIA HELD-OUT] Semilla 33


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

  Ep 1/8 | Loss 0.8984 | Train +1.52 | Held-Out +1.95 | Para +2.10
  Ep 2/8 | Loss 0.3510 | Train +0.35 | Held-Out +1.79 | Para +2.02
  Ep 3/8 | Loss 0.1373 | Train +0.03 | Held-Out +1.20 | Para +2.24
  Ep 4/8 | Loss 0.0939 | Train -0.03 | Held-Out +1.61 | Para +2.10
  Ep 5/8 | Loss 0.0889 | Train -0.03 | Held-Out +1.56 | Para +2.39
  Ep 6/8 | Loss 0.0520 | Train -0.07 | Held-Out +1.34 | Para +2.38
  Ep 7/8 | Loss 0.0586 | Train -0.18 | Held-Out +1.34 | Para +2.31
  Ep 8/8 | Loss 0.0649 | Train -0.11 | Held-Out +1.16 | Para +2.14
  [FINAL] PPL 46.0->59.8 | Held-Out +1.16

[RAMA BEATRIZ PYTHIA HELD-OUT] Semilla 33


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

  Ep 1/8 | Loss 0.6894 | Train +4.72 | Held-Out +2.30 | Para +1.87
  Ep 2/8 | Loss 0.0630 | Train +6.65 | Held-Out +2.20 | Para +1.54
  Ep 3/8 | Loss 0.0018 | Train +7.56 | Held-Out +1.95 | Para +1.38
  Ep 4/8 | Loss 0.0011 | Train +8.00 | Held-Out +1.85 | Para +1.32
  Ep 5/8 | Loss 0.0009 | Train +7.86 | Held-Out +1.81 | Para +1.12
  Ep 6/8 | Loss 0.0008 | Train +8.23 | Held-Out +1.86 | Para +1.13
  Ep 7/8 | Loss 0.0006 | Train +8.37 | Held-Out +1.92 | Para +1.01
  Ep 8/8 | Loss 0.0014 | Train +7.30 | Held-Out +2.00 | Para +1.24
  [FINAL] PPL 46.0->128.2 | Held-Out +2.00

EXP14 TERMINADO EXITOSAMENTE
Reporte: /kaggle/working/exp14_beatriz_pythia/exp14_pythia_results.json
SHA-256: 2d71e2a7bf24e6132f2f2e7304ec1a91e38e113763bee3029c17a9ef34c35f58
